# 第18章　アンサンブルと不確実性 ― 複数モデルで確実性を高める

**『本格実装 医療診断支援AI（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## アンサンブルを、重み空間で安く作る ― SWA

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
swa_model = AveragedModel(model)
swa_sched = SWALR(optimizer, swa_lr=0.02)      # 終盤は高めのLRで谷底を巡回
#   ↑0.02はSGD・大バッチ前提の値。Adamで基準LRが1e-4なら、まず1e-4前後から試す
for epoch in range(swa_start, epochs):
    train_one_epoch(model, optimizer)
    swa_sched.step()
    swa_model.update_parameters(model)         # 現在の重みを平均へ取り込む
update_bn(train_loader, swa_model,
          device=next(swa_model.parameters()).device)  # ★BN統計の再計算を忘れない

## アンサンブルを、一つのモデルへ畳む ― 知識蒸留

In [ ]:
import torch.nn.functional as F
T, alpha = 3.0, 0.5
def distill_loss(student_logits, teacher_logits, y):
    soft = F.kl_div(F.log_softmax(student_logits / T, 1),
                    F.softmax(teacher_logits / T, 1),
                    reduction="batchmean") * (T * T)   # 温度で軟化、勾配スケールを補正
    hard = F.cross_entropy(student_logits, y)          # 正解ラベルも併用
    return alpha * soft + (1 - alpha) * hard

## 学習し直さずに束ねる ― モデルマージ

In [ ]:
# greedy soup の骨格：検証が上がる重みだけを平均に足す
souped = load(models[0]); n = 1
for m in models[1:]:
    cand = average_weights(souped, m, n)          # 既存平均に候補を一つ足す
    if val_score(cand) >= val_score(souped):      # 改善するなら採用
        souped, n = cand, n + 1